# Amazon ML Challenge 2026: Business Entity Resolution Pipeline v2
## Multi-Source Matching with Token-Class Weighted Scoring & GPU Acceleration

This notebook implements an end-to-end Entity Resolution pipeline across noisy business records:
- **Source 1 (Reference Entities)**: `train_source1.tsv`
- **Source 2 & 3 (Candidate Pools)**: `train_source2.tsv`, `train_source3.tsv`
- **Ground Truth**: `train_ground_truth.tsv`

### Key Architecture Highlights in v2:
1. **Preserved Token Vocabularies (Requirement 2)**: Universal text cleaning preserves prefixes, suffixes, and generic business descriptors (e.g. `inc`, `llc`, `services`, `center`, `barbershop`) as naming words without loss of context.
2. **List-Based Word Columns (Requirements 3 & 4)**: Replaces multiple numbered word columns with a single `name_words` column and a single `address_words` column, each containing a Python list of splitted tokens.
3. **Layered Blocking Pipeline (Requirement 5)**:
   - **Stage 1 (Country Partition)**: Candidates strictly matched on country.
   - **Stage 2 (Name Overlap Gate)**: Any candidate sharing at least one word from S1's `name_words` list passes to the scoring stage.
4. **Token-Class Weighted Scoring Formula (Requirement 6)**:
   - **Prefix / Suffix Words**: **1.0x** weight (mitigates generic corporate form noise).
   - **Single-Character Words (`len == 1`, non-prefix/suffix)**: **2.0x** weight (preserves brand initials like `'b'`).
   - **Proper Brand Name Words (`len > 1`, non-prefix/suffix)**: **8.0x** weight (heavily prioritizes distinctive brand identity).
   - **State Match**: **4.0x** weight (geographic consistency).
   - **Address Word Matches ($	ext{len} > 1$)**: **4.0x** weight (physical location confirmation).
5. **Dual Blocking Engines**:
   - Vectorized CPU blocking (`block_candidates_cpu`).
   - PyTorch GPU-accelerated sparse tensor blocking (`block_candidates_gpu`) supporting CUDA, Apple Silicon MPS, and CPU fallback.
6. **Bounded Top-15 Selection & Pairwise Feature Preparation (Requirement 7)**:
   - Compresses 10.3M candidate rows to Top $K=15$ candidates per entity.
   - Builds pairwise ML comparison features with binary ground truth labels (`label = 1 / 0`) ready for GBDT model training.


In [ ]:
import os
import sys
import re
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
from unidecode import unidecode
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.feature_extraction.text import HashingVectorizer

# Set seed and verify PyTorch compute device
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
)
print(f"✓ Environment initialized successfully | PyTorch compute device: {device}")


### 1. Load Training Datasets (Requirement 1)
Loads tab-separated data files from the challenge repository. Set `SAMPLE_FRAC = None` for full dataset execution, or a float (e.g. `0.2`) for rapid development iteration.


In [ ]:
DATA_DIR = "data/student_resource/dataset/train"
SAMPLE_FRAC = None

print("Loading TSV files...")
s1 = pd.read_csv(os.path.join(DATA_DIR, "train_source1.tsv"), sep="\t")
s2 = pd.read_csv(os.path.join(DATA_DIR, "train_source2.tsv"), sep="\t")
s3 = pd.read_csv(os.path.join(DATA_DIR, "train_source3.tsv"), sep="\t")
gt = pd.read_csv(os.path.join(DATA_DIR, "train_ground_truth.tsv"), sep="\t")

if SAMPLE_FRAC is not None and SAMPLE_FRAC < 1.0:
    s1 = s1.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
    s2 = s2.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
    s3 = s3.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_FRAC*100:.0f}% of records for rapid iteration.")

print(f"✓ Source 1 (Reference) : {len(s1):,} records")
print(f"✓ Source 2 (Candidates): {len(s2):,} records")
print(f"✓ Source 3 (Candidates): {len(s3):,} records")
print(f"✓ Ground Truth Mappings: {len(gt):,} records")


### 2. State Extraction & Standardization Lookups
Constructs standardized state dictionary mappings for US (50 states + DC + abbreviations), India (36 states & union territories), and France. Extracts state information from raw addresses before cleaning to guarantee high recall.


In [ ]:
US_STATES = {
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'florida': 'FL', 'georgia': 'GA',
    'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA',
    'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS',
    'missouri': 'MO', 'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV', 'new hampshire': 'NH',
    'new jersey': 'NJ', 'new mexico': 'NM', 'new york': 'NY', 'north carolina': 'NC',
    'north dakota': 'ND', 'ohio': 'OH', 'oklahoma': 'OK', 'oregon': 'OR', 'pennsylvania': 'PA',
    'rhode island': 'RI', 'south carolina': 'SC', 'south dakota': 'SD', 'tennessee': 'TN',
    'texas': 'TX', 'utah': 'UT', 'vermont': 'VT', 'virginia': 'VA', 'washington': 'WA',
    'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY', 'district of columbia': 'DC'
}
US_LOOKUP = {k.lower(): v for k, v in US_STATES.items()}
for v in set(US_STATES.values()):
    US_LOOKUP[v.lower()] = v

INDIA_STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh", "Goa", "Gujarat",
    "Haryana", "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala", "Madhya Pradesh",
    "Maharashtra", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab",
    "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh", "Uttarakhand",
    "West Bengal", "Delhi", "Jammu and Kashmir", "Ladakh", "Puducherry", "Chandigarh",
    "Andaman and Nicobar Islands", "Dadra and Nagar Haveli and Daman and Diu", "Lakshadweep"
]
INDIA_LOOKUP = {s.lower(): s for s in INDIA_STATES}
INDIA_LOOKUP.update({
    'up': 'Uttar Pradesh', 'mp': 'Madhya Pradesh', 'ap': 'Andhra Pradesh', 'tn': 'Tamil Nadu',
    'wb': 'West Bengal', 'ka': 'Karnataka', 'mh': 'Maharashtra', 'dl': 'Delhi', 'nct': 'Delhi',
    'rj': 'Rajasthan', 'gj': 'Gujarat', 'ts': 'Telangana', 'hr': 'Haryana', 'pb': 'Punjab',
    'uk': 'Uttarakhand', 'orissa': 'Odisha', 'pondicherry': 'Puducherry'
})

FRANCE_REGIONS = [
    "Auvergne-Rhone-Alpes", "Bourgogne-Franche-Comte", "Bretagne", "Centre-Val de Loire",
    "Corse", "Grand Est", "Hauts-de-France", "Ile-de-France", "Normandie", "Nouvelle-Aquitaine",
    "Occitanie", "Pays de la Loire", "Provence-Alpes-Cote d'Azur"
]
FRANCE_LOOKUP = {r.lower(): r for r in FRANCE_REGIONS}

def extract_state_info(addr, country):
    """
    Extracts standardized state name/code from raw address text.
    Returns: (standardized_state_or_None, has_state_flag: 1 or 0)
    """
    if not isinstance(addr, str) or not addr.strip():
        return (None, 0)
    country = str(country).strip()
    parts = [p.strip() for p in addr.split(',') if p.strip()]
    check_tokens = []
    if len(parts) >= 2:
        check_tokens.extend([parts[-1], parts[-2]])
    elif len(parts) == 1:
        check_tokens.append(parts[0])
    if len(parts) >= 3:
        check_tokens.append(parts[0])

    if country == "US":
        for tok in check_tokens:
            cleaned_tok = re.sub(r'[^a-zA-Z\s]', '', tok).strip().lower()
            if cleaned_tok in US_LOOKUP:
                return (US_LOOKUP[cleaned_tok], 1)
        for k, v in US_LOOKUP.items():
            if len(k) > 2 and re.search(r'\b' + re.escape(k) + r'\b', addr, re.IGNORECASE):
                return (v, 1)
        match = re.search(r'\b([A-Z]{2})\b', addr)
        if match and match.group(1).lower() in US_LOOKUP:
            return (US_LOOKUP[match.group(1).lower()], 1)

    elif country == "India":
        for tok in check_tokens:
            cleaned_tok = re.sub(r'[^a-zA-Z\s]', '', tok).strip().lower()
            if cleaned_tok in INDIA_LOOKUP:
                return (INDIA_LOOKUP[cleaned_tok], 1)
        for k, v in INDIA_LOOKUP.items():
            if len(k) > 2 and re.search(r'\b' + re.escape(k) + r'\b', addr, re.IGNORECASE):
                return (v, 1)
        for tok in parts:
            c = tok.strip().lower()
            if c in INDIA_LOOKUP and len(c) == 2:
                return (INDIA_LOOKUP[c], 1)

    elif country == "France":
        for r_lower, r_full in FRANCE_LOOKUP.items():
            if r_lower in addr.lower():
                return (r_full, 1)

    return (None, 0)

print("✓ State extraction lookups and parser compiled successfully!")


### 3. Universal Text Hygiene & Prefix/Suffix Word Registry (Requirement 2)
Applies universal text normalization (URL stripping, handle stripping, possessive unwrapping `'s`, Unicode transliteration via `unidecode`, and symbol normalization) while **preserving all prefix, suffix, and generic business descriptors as naming words** (Requirement 2).
Also registers `PREFIX_SUFFIX_WORDS` for precision-weighted match scoring.


In [ ]:
# Universal Text Cleaning
def clean_universal(text):
    """
    Enhanced universal text hygiene:
    - Strips pipe/slash appended URLs and web metadata
    - Strips explicit URLs (http://, www.)
    - Strips leading handle/hashtag symbols (@, #, *)
    - Strips domain extensions (.com, .org, .net, etc.)
    - Strips possessives: e.g. "Orelee's" -> "Orelee" (prevents rogue standalone 's')
    - Transliterates Unicode scripts (Indic, accents) to Latin ASCII
    - Normalizes leetspeak / OCR digits inside letters (0 -> o, 1 -> l)
    - Unwraps parenthetical annotations: "(India)" -> "India"
    - Standardizes symbols (& -> and, + -> space) and whitespace
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. Pipe/slash appended URLs or metadata
    text = re.sub(r"\s*\|\s*(?:www\.|https?://|\S+\.(?:com|org|net|in|co|io))\S*", "", text, flags=re.IGNORECASE)
    # 2. Explicit URLs and prefixes
    text = re.sub(r"https?://\S+|www\.\S+", "", text, flags=re.IGNORECASE)
    # 3. Leading handles/hashtags
    text = re.sub(r"^\s*[@#*]+\s*", "", text)
    # 4. Domain extensions
    text = re.sub(r"\.(com|org|net|in|co|io|biz|info|us|edu|gov)\b", " ", text, flags=re.IGNORECASE)
    # 5. Possessives
    text = re.sub(r"['’]s\b", "", text, flags=re.IGNORECASE)
    # 6. Transliteration to ASCII
    text = unidecode(text).lower()
    # 7. OCR / leetspeak digits
    text = re.sub(r"(?<=[a-z])0(?=[a-z])", "o", text)
    text = re.sub(r"(?<=[a-z])1(?=[a-z])", "l", text)
    # 8. Parenthetical unwrapping
    text = re.sub(r"[\(\[\{]([^\)\]\}]+)[\)\]\}]", r" \1 ", text)
    # 9. Symbols & whitespace
    text = re.sub(r"\s*&\s*", " and ", text)
    text = re.sub(r"\s*\+\s*", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Spelling Harmonization Rules
SPELLING_HARMONIZATION = [
    (r"\blaxmi\b", "lakshmi"),
    (r"\b(choudhary|choudhury|chowdhury)\b", "chaudhary"),
    (r"\bbikas\b", "vikas"),
    (r"\bparsad\b", "prasad"),
]

# Comprehensive Prefix, Suffix & Generic Business Descriptor Registry
# In v2, these tokens are KEPT in clean_name and receive 1.0x weight in scoring
PREFIX_SUFFIX_WORDS = {
    # 1. Corporate & Legal Forms
    "private", "limited", "pvt", "ltd", "praaivett", "limittedd", "elelpii",
    "liability", "company", "llc", "pllc", "llp", "incorporated", "corporation", "inc", "corp",
    "co", "public", "foundation", "trust", "sarl", "sas", "sa", "eurl", "snc",
    # 2. Domains / Handles
    "com", "org", "net",
    # 3. Commercial & Entity Descriptors
    "enterprises", "enterprise", "industries", "industry", "solutions", "services", "service", "svc", "serv",
    "holdings", "holding", "group", "grp", "associates", "assoc", "brothers", "bros", "sons", "son", "partners",
    "ventures", "capital", "trading", "traders", "exports", "marketing", "management", "mgmt",
    "products", "agency", "firm",
    # 4. Tech, Engineering & Infrastructure
    "technologies", "technology", "tech", "techno", "systems", "system", "engineering", "engg", "infratech", "infra",
    # 5. Consulting & Advisory
    "consulting", "consultancy", "consultants", "consultant", "advisory", "advisors",
    # 6. Healthcare Descriptors
    "medicine", "medical", "clinic", "care", "health", "healthcare", "dental", "hospital", "pediatric", "specialists", "physicians", "pharmaceuticals", "pharma",
    # 7. Retail & Facility Descriptors
    "center", "centre", "business", "store", "shop", "barbershop", "salon", "bazaar", "market", "mart",
    # 8. Education & Institutions
    "institute", "institution", "school", "academy", "society",
    # 9. Geographic / Scope Descriptors
    "international", "intl", "int", "global", "national", "metro", "india",
    # 10. Honorifics
    "sri", "shri", "smt",
    # 11. Generic Prepositions & Stopwords
    "the", "and", "of", "in", "at", "for", "to", "a", "an"
}

def normalize_name(name):
    """
    Cleans business name while retaining all prefix, suffix, and generic words.
    Does NOT remove corporate forms or suffixes.
    """
    clean = clean_universal(name)
    for pattern, repl in SPELLING_HARMONIZATION:
        clean = re.sub(pattern, repl, clean)
    clean = re.sub(r"\s+", " ", clean).strip()
    return clean

# Address Standardization Expansions
ADDR_EXPANSIONS = [
    (r"\b(st|str)\b", "street"),
    (r"\b(rd|marg|rasta)\b", "road"),
    (r"\b(ave|av)\b", "avenue"),
    (r"\bblvd\b", "boulevard"),
    (r"\bhwy\b", "highway"),
    (r"\bdr\b", "drive"),
    (r"\bln\b", "lane"),
    (r"\bct\b", "court"),
    (r"\bpkwy\b", "parkway"),
    (r"\bn\b", "north"),
    (r"\bs\b", "south"),
    (r"\be\b", "east"),
    (r"\bw\b", "west"),
    (r"\bne\b", "northeast"),
    (r"\bnw\b", "northwest"),
    (r"\bse\b", "southeast"),
    (r"\bsw\b", "southwest"),
    (r"\b(ste|ste#|suite#)\b", "suite"),
    (r"\b(apt|flat)\b", "apartment"),
    (r"\b(fl|flr)\b", "floor"),
    (r"\b(gr flr|gf)\b", "ground floor"),
    (r"\bbldg\b", "building"),
    (r"\b(plt|plot no)\b", "plot"),
    (r"\b(sec|sector no)\b", "sector"),
    (r"\b(opp|opp to|opposite to)\b", "opposite"),
    (r"\b(nr|near to)\b", "near"),
    (r"\bbehind\b", "behind"),
]

def normalize_address(addr):
    """Address standardization across abbreviations, landmarks, and street types."""
    clean = clean_universal(addr)
    for pattern, repl in ADDR_EXPANSIONS:
        clean = re.sub(pattern, repl, clean)
    return re.sub(r"\s+", " ", clean).strip()

print(f"✓ Normalization pipeline defined | Prefix/suffix words registered: {len(PREFIX_SUFFIX_WORDS):,}")


### 4. Data Preprocessing & Word List Column Creation (Requirements 3 & 4)
Processes $S_1$, $S_2$, and $S_3$ in-place to create:
1. `name_words`: A **single column containing the list of splitted words** from `clean_name` (no multiple numbered columns).
2. `clean_address`: Standardized address.
3. `address_words`: A **single column containing the list of splitted words** from `clean_address`.
4. `state` & `has_state`: Extracted geographic state metadata.


In [ ]:
def add_features_in_place(df, name="dataset"):
    """
    Extracts states, cleans text, and creates single list-of-words columns:
    - name_words: list of splitted tokens from clean_name (retaining prefixes/suffixes)
    - address_words: list of splitted tokens from clean_address
    Modifies DataFrame in-place without high-overhead copies.
    """
    print(f"\n--- Adding features in-place to {name} ({len(df):,} records) ---")

    # 1. State Extraction before text cleaning
    print(f"[{name}] Extracting states from raw address...")
    state_tuples = [extract_state_info(addr, country) for addr, country in zip(df["business_address"], df["country"])]
    df["state"] = [t[0] for t in state_tuples]
    df["has_state"] = [t[1] for t in state_tuples]

    # 2. Business Name Cleaning & Word List Creation (Requirement 2 & 3)
    tqdm.pandas(desc=f"[{name}] Clean Names")
    df["clean_name"] = df["business_name"].fillna("").astype(str).progress_apply(normalize_name)
    df["name_words"] = df["clean_name"].apply(lambda s: [w for w in s.split() if len(w) >= 1])

    # 3. Business Address Cleaning & Word List Creation (Requirement 4)
    tqdm.pandas(desc=f"[{name}] Clean Addresses")
    df["clean_address"] = df["business_address"].fillna("").astype(str).progress_apply(normalize_address)
    df["address_words"] = df["clean_address"].apply(lambda s: [w for w in s.split() if len(w) >= 1])

    return df

# Execute preprocessing on s1, s2, s3 in-place
add_features_in_place(s1, "Source 1 (Ref)")
add_features_in_place(s2, "Source 2 (Cands)")
add_features_in_place(s3, "Source 3 (Cands)")

print("\n✓ Preprocessing completed! Displaying sample S1 reference records with list columns:")
display(s1[["entity_id", "country", "business_name", "clean_name", "name_words", "state", "address_words"]].head(5))


### 5. Layered Candidate Blocking & Token-Class Scoring (Requirements 5, 6 & 7)
Implements candidate filtering and scoring in three sequential stages:

1. **Stage 1 (Country Partitioning)**:
   - Candidates must strictly match the reference entity's country (`candidate['country'] == s1['country']`). Eliminates cross-border false matches.
2. **Stage 2 (Name Overlap Gate)**:
   - The candidate's `name_words` list must share **at least one token** with the $S_1$ entity's `name_words` list (or match the concatenated brand handle). Any record with zero name overlap is excluded.
3. **Stage 3 (Token-Class Weighted Scoring Formula)**:
   $$\mathbf{Score} = 1.0 \times \text{Overlap}_{\text{prefix/suffix}} + 2.0 \times \text{Overlap}_{\text{len}=1} + 8.0 \times \text{Overlap}_{\text{proper brand}} + 4.0 \times \text{Overlap}_{\text{address}} + 4.0 \times \text{State Match}$$
   - **Prefix / Suffix Words** (weight **1.0x**): Words in `PREFIX_SUFFIX_WORDS` (e.g. `inc`, `llc`, `services`, `center`).
   - **Single-Character Words** (weight **2.0x**): Single-letter tokens not in prefix/suffix (e.g. initial `'b'`).
   - **Proper Brand Name Words** (weight **8.0x**): Multi-character distinctive brand tokens not in prefix/suffix (e.g. `'orelee'`, `'retail'`, `'prime'`, `'money'`).
   - **Address Word Matches** (weight **4.0x**): Overlapping words between `address_words` (length $> 1$).
   - **State Match** (weight **4.0x**): Exact match between extracted state metadata.
4. **Top-15 Bounded Selection**:
   - Ranks passing candidates by `matching_score` descending and retains the Top **15** records per $S_1$ entity.


In [ ]:
def block_candidates_cpu(s1_subset, s2_df, s3_df, top_k=15):
    """
    CPU Reference Implementation of Layered Candidate Blocking & Token-Class Scoring.
    """
    blocked_candidates = {}
    blocking_summary = []

    cands_combined = pd.concat([
        s2_df.assign(source="S2"),
        s3_df.assign(source="S3")
    ], ignore_index=True)

    for _, s1_row in s1_subset.iterrows():
        sid = s1_row["entity_id"]
        scountry = s1_row["country"]
        s_words = s1_row["name_words"]
        s_word_set = set(s_words)
        s_joined = "".join(s_words) if len(s_words) >= 2 else None
        if s_joined:
            s_word_set.add(s_joined)

        s_addr_words = set(w for w in s1_row["address_words"] if len(w) > 1)
        s_state = s1_row.get("state")
        s_has_st = s1_row.get("has_state", 1 if pd.notna(s_state) else 0)

        # Stage 1: Country filter
        c_sub = cands_combined[cands_combined["country"] == scountry]
        if len(c_sub) == 0:
            blocked_candidates[sid] = pd.DataFrame()
            continue

        candidate_scores = []
        for _, c_row in c_sub.iterrows():
            c_words = c_row["name_words"]
            c_word_set = set(c_words)

            # Stage 2: Name overlap gate
            overlap = s_word_set.intersection(c_word_set)
            if not overlap and s_joined and (s_joined in c_word_set or s_joined == c_row["clean_name"]):
                overlap = set(s_words)

            if not overlap:
                continue

            # Stage 3: Token-Class Weighted Scoring
            w_pref = sum(1 for w in overlap if w in PREFIX_SUFFIX_WORDS)
            w_l1 = sum(1 for w in overlap if len(w) == 1 and w not in PREFIX_SUFFIX_WORDS)
            w_prop = sum(1 for w in overlap if len(w) > 1 and w not in PREFIX_SUFFIX_WORDS)

            c_addr_words = set(w for w in c_row["address_words"] if len(w) > 1)
            a_overlap = len(s_addr_words.intersection(c_addr_words))

            c_state = c_row.get("state")
            c_has_st = c_row.get("has_state", 1 if pd.notna(c_state) else 0)
            st_match = 1 if (s_has_st == 1 and c_has_st == 1 and s_state == c_state) else 0

            score = 1.0 * w_pref + 2.0 * w_l1 + 8.0 * w_prop + 4.0 * a_overlap + 4.0 * st_match

            candidate_scores.append({
                "entity_id": c_row["entity_id"],
                "source": c_row["source"],
                "country": c_row["country"],
                "business_name": c_row["business_name"],
                "clean_name": c_row["clean_name"],
                "business_address": c_row["business_address"],
                "clean_address": c_row["clean_address"],
                "state": c_state,
                "state_match": st_match,
                "prefix_suffix_overlap": w_pref,
                "len1_overlap": w_l1,
                "proper_brand_overlap": w_prop,
                "name_overlap_count": len(overlap),
                "address_overlap_count": a_overlap,
                "matching_score": float(score),
                "s1_entity_id": sid
            })

        if candidate_scores:
            cands_df = pd.DataFrame(candidate_scores)
            cands_df = cands_df.sort_values(by="matching_score", ascending=False).drop_duplicates("entity_id").head(top_k).reset_index(drop=True)
            blocked_candidates[sid] = cands_df
        else:
            blocked_candidates[sid] = pd.DataFrame()

        n_c = len(blocked_candidates[sid])
        blocking_summary.append({
            "s1_entity_id": sid,
            "business_name": s1_row["business_name"],
            "country": scountry,
            "state": s_state,
            "total_candidates": n_c,
            "max_score": float(blocked_candidates[sid]["matching_score"].max()) if n_c > 0 else 0.0,
            "avg_score": round(float(blocked_candidates[sid]["matching_score"].mean()), 2) if n_c > 0 else 0.0,
            "state_matches": int(blocked_candidates[sid]["state_match"].sum()) if n_c > 0 else 0
        })

    return blocked_candidates, pd.DataFrame(blocking_summary)

print("✓ CPU candidate blocking function defined successfully!")


### 6. GPU-Accelerated PyTorch Tensor Blocking (`block_candidates_gpu`)
High-performance GPU blocking engine utilizing PyTorch Sparse Matrix Multiplication (**SpMM**) on CUDA / MPS:
- Generates high-dimensional sparse representations via `HashingVectorizer(n_features=2**19)` (524,288 features, zero dictionary storage).
- Builds weighted dense query tensors for $S_1$ batches ($B=100$) embedding the exact token-class weights ($1	imes, 2	imes, 8	imes$).
- Streams candidate pools in chunks of 100,000 rows as PyTorch Sparse COO Tensors (`torch.sparse_coo_tensor`, $< 40$ MB VRAM per chunk).
- Computes token overlap matrix dot products in $< 1$ ms on GPU.
- Applies Stage 2 Name Overlap Gate (`name_score > 0`) and ranks Top $K=15$ on GPU via `torch.topk`.


In [ ]:
def block_candidates_gpu(
    s1_subset, 
    s2_df, 
    s3_df, 
    s1_batch_size=100, 
    cand_chunk_size=100000, 
    top_k=15, 
    n_features=2**19, 
    device=None
):
    """
    GPU-Accelerated Candidate Blocking & Token-Class Scoring using PyTorch Sparse Tensors.
    
    Pipeline v2 Scoring Formula:
      score = 1.0 * prefix_suffix + 2.0 * len1 + 8.0 * proper_brand + 4.0 * address + 4.0 * state
      Gate: requires name_scores > 0 (at least one shared name token)
      
    Architecture & Memory Optimization (Precomputed Candidate CSR + Bounded VRAM):
      1. Pre-vectorizes candidate chunks into Scipy CSR matrices ONCE on CPU (uses ~350 MB CPU RAM, 0 MB GPU VRAM).
      2. Streams S1 batches with Bounded GPU Memory (< 1.0 GB peak VRAM):
         - Only ONE batch of dense queries (batch_size=100, ~600 MB) resides on GPU at any time.
         - Sparse candidate tensors are streamed to GPU and freed immediately per chunk.
         - Calls torch.cuda.empty_cache() after every S1 batch to completely eliminate CUDA fragmentation and OOM.
      3. Stage 2 Name Overlap Gate: requires name_scores > 0, excluding candidates with 0 name overlap.
      4. Uses torch.topk(..., k=15) on GPU to retain Top-15 highest-scoring candidates per S1 entity.
    """
    if device is None:
        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
    print(f"--- Running GPU Candidate Blocking on Device: {device} ---")

    # 1. Hashing Vectorizers for Name and Address
    name_vec = HashingVectorizer(n_features=n_features, token_pattern=r"(?u)\b\w+\b", alternate_sign=False, norm=None, binary=True)
    addr_vec = HashingVectorizer(n_features=n_features, token_pattern=r"(?u)\b\w{2,}\b", alternate_sign=False, norm=None, binary=True)

    # State ID mapping
    all_states = sorted(list(set(s1_subset["state"].dropna().unique())))
    state_to_id = {s: i for i, s in enumerate(all_states)}

    def scipy_to_torch_sparse(scipy_mat, target_dev):
        coo = scipy_mat.tocoo()
        if coo.nnz == 0:
            idx = torch.empty((2, 0), dtype=torch.long)
            val = torch.empty(0, dtype=torch.float32)
        else:
            idx = torch.from_numpy(np.vstack((coo.row, coo.col))).long()
            val = torch.from_numpy(coo.data).float()
        return torch.sparse_coo_tensor(idx, val, torch.Size(coo.shape), device=target_dev).coalesce()

    top_candidates = {sid: pd.DataFrame() for sid in s1_subset["entity_id"]}

    # 2. Partition candidate pools by country once to eliminate cross-country overhead
    cand_pools_by_country = {}
    for c in s1_subset["country"].dropna().unique():
        cols_s2 = [col for col in ["entity_id", "country", "business_name", "clean_name", "name_words", "business_address", "clean_address", "address_words", "state"] if col in s2_df.columns]
        s2_sub = s2_df[s2_df["country"] == c][cols_s2].copy()
        s2_sub["source"] = "S2"

        cols_s3 = [col for col in ["entity_id", "country", "business_name", "clean_name", "name_words", "business_address", "clean_address", "address_words", "state"] if col in s3_df.columns]
        s3_sub = s3_df[s3_df["country"] == c][cols_s3].copy()
        s3_sub["source"] = "S3"
        cand_pools_by_country[c] = pd.concat([s2_sub, s3_sub], ignore_index=True)

    n_s1 = len(s1_subset)
    print(f"Total S1 Reference Entities: {n_s1:,} | Batch Size: {s1_batch_size}")

    for country, cand_pool in cand_pools_by_country.items():
        s1_country = s1_subset[s1_subset["country"] == country].reset_index(drop=True)
        n_s1_country = len(s1_country)
        n_cands = len(cand_pool)
        if n_s1_country == 0 or n_cands == 0:
            continue

        # 3. Pre-vectorize candidate chunks ONCE into CPU Scipy CSR matrices (Uses 0 MB GPU VRAM)
        chunk_starts = list(range(0, n_cands, cand_chunk_size))
        cand_chunks_precomputed = []
        pbar_pre = tqdm(chunk_starts, desc=f"↳ [CPU] Pre-vectorizing {country} Candidates ({n_cands:,})", leave=False)
        for c_start in pbar_pre:
            c_end = min(c_start + cand_chunk_size, n_cands)
            c_chunk = cand_pool.iloc[c_start:c_end].reset_index(drop=True)

            c_name_csr = name_vec.transform(c_chunk["clean_name"].fillna("").astype(str))
            c_addr_csr = addr_vec.transform(c_chunk["clean_address"].fillna("").astype(str))
            cand_st_ids_np = np.array([state_to_id.get(s, -2) if pd.notna(s) else -2 for s in c_chunk["state"]], dtype=np.int16)

            cand_chunks_precomputed.append((c_chunk, c_name_csr, c_addr_csr, cand_st_ids_np))

        # 4. Stream S1 batches with Bounded GPU Memory (< 1.0 GB peak VRAM)
        s1_batches = list(range(0, n_s1_country, s1_batch_size))
        pbar_s1 = tqdm(s1_batches, desc=f"↳ [GPU] Matching {country} Entities ({n_s1_country:,} queries)", leave=True)

        for s1_start in pbar_s1:
            s1_end = min(s1_start + s1_batch_size, n_s1_country)
            s1_group = s1_country.iloc[s1_start:s1_end].reset_index(drop=True)
            group_s1_ids = list(s1_group["entity_id"])
            group_size = len(s1_group)

            # Build S1 query matrices with token-class weights (1x prefix/suffix, 2x len1, 8x proper brand)
            q_name_rows = []
            for _, r in s1_group.iterrows():
                words = r["name_words"]
                pref = " ".join([w for w in words if w in PREFIX_SUFFIX_WORDS])
                l1 = " ".join([w for w in words if len(w) == 1 and w not in PREFIX_SUFFIX_WORDS])
                prop = " ".join([w for w in words if len(w) > 1 and w not in PREFIX_SUFFIX_WORDS])
                joined = "".join(words) if len(words) >= 2 else ""
                if joined:
                    prop = (prop + " " + joined).strip()

                v_pref = name_vec.transform([pref]) * 1.0 if pref else None
                v_l1 = name_vec.transform([l1]) * 2.0 if l1 else None
                v_prop = name_vec.transform([prop]) * 8.0 if prop else None

                v_comb = v_pref if v_pref is not None else name_vec.transform([""]) * 0.0
                if v_l1 is not None:
                    v_comb = v_comb + v_l1
                if v_prop is not None:
                    v_comb = v_comb + v_prop
                q_name_rows.append(v_comb.toarray()[0])

            # Move ONLY this current batch to GPU (~600 MB)
            q_name_dense = torch.from_numpy(np.array(q_name_rows)).float().to(device)
            q_addr_dense = torch.from_numpy((addr_vec.transform(s1_group["clean_address"].fillna("").astype(str)) * 4.0).toarray()).float().to(device)
            s1_st_ids = torch.tensor([state_to_id.get(s, -1) if pd.notna(s) else -1 for s in s1_group["state"]], device=device, dtype=torch.int16)

            running_cands = [[] for _ in range(group_size)]

            for c_chunk, c_name_csr, c_addr_csr, cand_st_ids_np in cand_chunks_precomputed:
                c_name_sp = scipy_to_torch_sparse(c_name_csr, device)
                c_addr_sp = scipy_to_torch_sparse(c_addr_csr, device)
                cand_st_ids = torch.from_numpy(cand_st_ids_np).to(device)

                # Overlap SpMM
                name_scores = torch.sparse.mm(c_name_sp, q_name_dense.t())
                addr_scores = torch.sparse.mm(c_addr_sp, q_addr_dense.t())

                # State match (4.0x)
                state_match = (cand_st_ids.unsqueeze(1) == s1_st_ids.unsqueeze(0)) & (s1_st_ids.unsqueeze(0) >= 0)
                state_scores = state_match.float() * 4.0

                total_scores = name_scores + addr_scores + state_scores

                # Stage 2 Name Overlap Gate: requires name_scores > 0
                has_name_overlap = (name_scores > 0)
                total_scores = torch.where(has_name_overlap, total_scores, torch.tensor(-1.0, device=device))

                for g_idx in range(group_size):
                    ent_scores = total_scores[:, g_idx]
                    pos_mask = ent_scores > 0
                    if not pos_mask.any():
                        continue
                    pos_idx = torch.nonzero(pos_mask).squeeze(1)
                    pos_scores = ent_scores[pos_idx]

                    k_val = min(top_k, len(pos_scores))
                    top_vals, top_k_subidx = torch.topk(pos_scores, k=k_val)
                    cand_indices = pos_idx[top_k_subidx].cpu().numpy()

                    ent_st_mat = (state_scores[:, g_idx][cand_indices] > 0).cpu().numpy().astype(int)
                    ent_addr_ov = (addr_scores[:, g_idx][cand_indices] / 4.0).cpu().numpy().astype(int)

                    for sc, c_idx, a_ov, st_m in zip(top_vals.cpu().numpy(), cand_indices, ent_addr_ov, ent_st_mat):
                        r = c_chunk.iloc[c_idx]
                        running_cands[g_idx].append({
                            "entity_id": r["entity_id"],
                            "source": r["source"],
                            "country": r["country"],
                            "business_name": r["business_name"],
                            "clean_name": r["clean_name"],
                            "name_words": r.get("name_words", []),
                            "business_address": r.get("business_address", ""),
                            "clean_address": r["clean_address"],
                            "address_words": r.get("address_words", []),
                            "state": r.get("state"),
                            "state_match": int(st_m),
                            "address_overlap_count": int(a_ov),
                            "matching_score": float(sc)
                        })

                # Explicitly free chunk tensors immediately
                del c_name_sp, c_addr_sp, cand_st_ids, name_scores, addr_scores, state_scores, total_scores

            # Explicitly free batch query tensors from GPU
            del q_name_dense, q_addr_dense, s1_st_ids
            if device.type == "cuda":
                torch.cuda.empty_cache()

            # Finalize Top-K per S1 entity for this batch
            for g_idx, sid in enumerate(group_s1_ids):
                if not running_cands[g_idx]:
                    top_candidates[sid] = pd.DataFrame()
                    continue
                cands_df = pd.DataFrame(running_cands[g_idx])
                cands_df = cands_df.sort_values(by="matching_score", ascending=False).drop_duplicates("entity_id").head(top_k).reset_index(drop=True)
                cands_df["s1_entity_id"] = sid
                top_candidates[sid] = cands_df

        del cand_chunks_precomputed

    blocking_summary = []
    for _, s1_row in s1_subset.iterrows():
        sid = s1_row["entity_id"]
        cands = top_candidates[sid]
        n_c = len(cands)
        blocking_summary.append({
            "s1_entity_id": sid,
            "business_name": s1_row["business_name"],
            "country": s1_row["country"],
            "state": s1_row.get("state"),
            "total_candidates": n_c,
            "max_score": float(cands["matching_score"].max()) if n_c > 0 else 0.0,
            "avg_score": round(float(cands["matching_score"].mean()), 2) if n_c > 0 else 0.0,
            "state_matches": int(cands["state_match"].sum()) if n_c > 0 and "state_match" in cands else 0
        })

    return top_candidates, pd.DataFrame(blocking_summary)

# Execute GPU blocking preview on first 10 S1 records
s1_top10 = s1.head(10)
blocked_cands_gpu, summary_df_gpu = block_candidates_gpu(
    s1_top10, s2, s3, 
    s1_batch_size=10, 
    cand_chunk_size=100000, 
    top_k=15
)

print("=== GPU Candidate Blocking & Token-Class Ranking Summary (First 10 S1 Records) ===")
display(summary_df_gpu)

# Sample preview of top blocked candidates for first entity
for s1_id in s1_top10["entity_id"]:
    if len(blocked_cands_gpu[s1_id]) > 0:
        first_s1_name = s1_top10[s1_top10["entity_id"] == s1_id]["business_name"].values[0]
        print(f"\n=== Top GPU Blocked Candidates for {s1_id} ({first_s1_name}) ===")
        display(blocked_cands_gpu[s1_id][["entity_id", "source", "country", "business_name", "clean_name", "state", "state_match", "address_overlap_count", "matching_score"]].head(5))
        break


### 7. Ground Truth Recovery & Blocking Recall Evaluation
Quantifies candidate blocking effectiveness by benchmarking blocked candidate pools against `train_ground_truth.tsv`:
- **Target Recovery Rate (Recall @ 15)**: Percentage of true target matches retained in the Top 15 candidate pool.
- **Entity Hit Rate**: Percentage of reference entities where at least 1 true match was recovered.
- **Full Recovery Rate**: Percentage of reference entities where 100% of all true matches were captured.
- **Mean Reciprocal Rank (MRR)**: Average reciprocal rank $\frac{1}{\text{rank}_1}$ of the first true match.
- **Rank Distribution Breakdown**: Detailed distribution across Rank 1, Ranks 2–5, Ranks 6–10, Ranks 11–15, and Missed.


In [ ]:
def evaluate_blocking_recall(s1_subset, candidate_dict, gt_df, method_name=None):
    """
    Evaluates Ground Truth Recovery & Recall after candidate blocking.
    Compares candidate pools against true matches in train_ground_truth.tsv.
    """
    gt_map = {}
    s1_ids = set(s1_subset['entity_id'])
    gt_sub = gt_df[gt_df['source1_entity_id'].isin(s1_ids)]
    for _, row in gt_sub.iterrows():
        gt_map[row['source1_entity_id']] = [x.strip() for x in str(row['matched_entity_ids']).split(',') if x.strip()]

    eval_rows = []
    total_gt = 0
    total_recovered = 0
    rank_dist = {'Rank 1': 0, 'Ranks 2-5': 0, 'Ranks 6-10': 0, 'Ranks 11-15': 0, 'Missed': 0}
    reciprocal_ranks = []

    for _, s1_row in s1_subset.iterrows():
        s1_id = s1_row['entity_id']
        true_targets = gt_map.get(s1_id, [])
        num_gt = len(true_targets)
        total_gt += num_gt

        cands_df = candidate_dict.get(s1_id, pd.DataFrame())
        cand_id_list = list(cands_df['entity_id']) if len(cands_df) > 0 and 'entity_id' in cands_df.columns else []

        found_ranks = []
        for tgt in true_targets:
            if tgt in cand_id_list:
                r = cand_id_list.index(tgt) + 1
                found_ranks.append(r)
                if r == 1:
                    rank_dist['Rank 1'] += 1
                elif r <= 5:
                    rank_dist['Ranks 2-5'] += 1
                elif r <= 10:
                    rank_dist['Ranks 6-10'] += 1
                elif r <= 15:
                    rank_dist['Ranks 11-15'] += 1
            else:
                rank_dist['Missed'] += 1

        rec_count = len(found_ranks)
        total_recovered += rec_count
        rec_pct = round((rec_count / num_gt * 100), 1) if num_gt > 0 else 0.0

        first_r = min(found_ranks) if found_ranks else None
        if first_r:
            reciprocal_ranks.append(1.0 / first_r)
        else:
            reciprocal_ranks.append(0.0)

        status = '✓ Full' if (rec_count == num_gt and num_gt > 0) else ('✓ Partial' if rec_count > 0 else '✗ Missed')
        eval_rows.append({
            's1_entity_id': s1_id,
            'business_name': s1_row.get('business_name', ''),
            'country': s1_row.get('country', ''),
            'gt_matches': num_gt,
            'recovered_in_top_15': rec_count,
            'recall_rate': f'{rec_pct:.1f}%',
            'ranks_of_matches': sorted(found_ranks),
            'first_match_rank': first_r if first_r else '-',
            'status': status
        })

    res_df = pd.DataFrame(eval_rows)
    overall_recall = (total_recovered / total_gt * 100) if total_gt > 0 else 0.0
    hit_rate = (sum(1 for r in eval_rows if r['recovered_in_top_15'] > 0) / len(eval_rows) * 100) if eval_rows else 0.0
    full_recall_rate = (sum(1 for r in eval_rows if r['recovered_in_top_15'] == r['gt_matches']) / len(eval_rows) * 100) if eval_rows else 0.0
    mrr = float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0

    header = f"GROUND TRUTH RECOVERY & BLOCKING RECALL EVALUATION ({method_name})" if method_name else "GROUND TRUTH RECOVERY & BLOCKING RECALL EVALUATION"
    print("=" * 75)
    print(header)
    print("=" * 75)
    print(f"Total S1 Entities Evaluated   : {len(s1_subset):,}")
    print(f"Total Ground Truth Targets    : {total_gt:,}")
    print(f"Total Targets Recovered (Top15): {total_recovered:,} / {total_gt:,} ({overall_recall:.2f}%)")
    print(f"Entity Hit Rate (≥ 1 match)      : {hit_rate:.1f}%")
    print(f"Full Recovery Rate (100% GT)  : {full_recall_rate:.1f}%")
    print(f"Mean Reciprocal Rank (MRR)    : {mrr:.4f}")
    print("\nRank Distribution of True Matches:")
    for band, count in rank_dist.items():
        pct = (count / total_gt * 100) if total_gt > 0 else 0.0
        print(f"  - {band:<14} : {count:>3} matches ({pct:>5.1f}%)")
    print("=" * 75)

    return res_df

# Evaluate GPU-accelerated candidate pool against Ground Truth
print(">>> EVALUATING GPU TENSOR BLOCKING (Top 15 per Entity) <<<")
blocking_eval_df = evaluate_blocking_recall(s1_top10, blocked_cands_gpu, gt, method_name="GPU Tensor Blocking v2")
display(blocking_eval_df)


### 8. Post-Blocking Pairwise Feature Engineering & Training Data Preparation
Transforms blocked candidate pairs `(s1_entity, candidate)` into a structured **comparison matrix** with attached binary target labels (`label = 1` for True Match, `0` for Negative Candidate):
1. **Pairwise Mapping ($1 : 15$)**: Each $S_1$ entity maps to its Top 15 blocked candidates ($10 \times 15 = 150$ candidate pairs for the sample).
2. **Comparison Features**:
   - `blocking_score`: Multi-signal weighted score from candidate blocking.
   - `common_words_count`, `word_jaccard`: Word token overlap and Jaccard similarity across `name_words`.
   - `is_exact_core_match`, `is_substring_match`: Exact clean name match and substring inclusion flags.
   - `has_state_s1`, `has_state_cand`, `both_have_state`, `state_match`: State metadata alignment.
   - `has_house_num_s1`, `has_house_num_cand`, `both_have_house_num`, `house_number_match`: Physical building/plot number regex matching.
   - `address_common_words_count`, `address_word_jaccard`: Overlapping words across `address_words` and Jaccard similarity.
3. **Binary Label Attachment**: Attaches `label = 1` if `candidate_id` is in `train_ground_truth.tsv`, else `0`.


In [ ]:
# House Number Regex Extractor
def extract_house_number(address):
    """
    Extracts street, building, plot, or house number from raw address.
    Returns: (house_number_str, has_house_number_flag)
    """
    if not isinstance(address, str) or not address.strip():
        return (None, 0)
    street_num = re.search(
        r'\b(\d+[a-zA-Z]?(?:[-/]\d+)?)\s+(?:[a-zA-Z\.]+\s+)*(?:street|st|road|rd|avenue|ave|drive|dr|boulevard|blvd|lane|ln|court|ct|parkway|pkwy|highway|hwy|marg|circle|cir|way)\b',
        address, re.IGNORECASE
    )
    if street_num:
        return (street_num.group(1).lower(), 1)
    explicit = re.search(r'(?:#|no\.?|plot\s*(?:no\.?)?|shop\s*(?:no\.?)?|flat\s*(?:no\.?)?)\s*([0-9]+[a-zA-Z]?(?:[-/][0-9]+[a-zA-Z]?)?)', address, re.IGNORECASE)
    if explicit:
        return (explicit.group(1).lower(), 1)
    leading = re.match(r'^\s*([0-9]+[a-zA-Z]?(?:[-/][0-9]+[a-zA-Z]?)?)\b', address)
    if leading:
        return (leading.group(1).lower(), 1)
    nums = re.findall(r'\b([0-9]{1,5}[a-zA-Z]?)\b', address)
    if nums:
        return (nums[0].lower(), 1)
    return (None, 0)

# Extract house numbers for S1 in-place
if 'house_number' not in s1.columns:
    print("Extracting house numbers for S1 reference records...")
    hn_tuples = [extract_house_number(addr) for addr in s1['business_address']]
    s1['house_number'] = [t[0] for t in hn_tuples]
    s1['has_house_number'] = [t[1] for t in hn_tuples]

def compute_pair_features(s1_df, candidate_dict, s2_df=None, s3_df=None, gt_df=None):
    """
    Generates pairwise comparison features for all candidate pairs (s1_entity, candidate).
    If gt_df is provided, attaches binary classification target 'label' (1=True Match, 0=False).
    """
    feature_rows = []

    gt_map = {}
    if gt_df is not None:
        s1_ids = set(s1_df['entity_id'])
        gt_sub = gt_df[gt_df['source1_entity_id'].isin(s1_ids)]
        for _, row in gt_sub.iterrows():
            gt_map[row['source1_entity_id']] = set(x.strip() for x in str(row['matched_entity_ids']).split(',') if x.strip())

    for _, s1_row in s1_df.iterrows():
        s1_id = s1_row['entity_id']
        cands = candidate_dict.get(s1_id)
        if cands is None or len(cands) == 0:
            continue

        s1_clean = str(s1_row.get('clean_name', s1_row['business_name']))
        s1_words = set(s1_row['name_words'])
        s1_addr = str(s1_row.get('clean_address', s1_row['business_address']))
        s1_addr_words = set(w for w in s1_row['address_words'] if len(w) > 1)
        s1_state = s1_row.get('state')
        s1_has_state = s1_row.get('has_state', 1 if pd.notna(s1_state) else 0)

        s1_hn, s1_has_hn = s1_row.get('house_number'), s1_row.get('has_house_number', 0)
        if s1_hn is None:
            s1_hn, s1_has_hn = extract_house_number(s1_row['business_address'])

        true_targets = gt_map.get(s1_id, set())

        for _, cand_row in cands.iterrows():
            cand_id = cand_row['entity_id']
            cand_src = cand_row.get('source', 'S2')
            cand_clean = str(cand_row.get('clean_name', cand_row.get('business_name', '')))
            cand_words = set(cand_row.get('name_words', cand_clean.split()))

            cand_addr = str(cand_row.get('clean_address', cand_row.get('business_address', '')))
            cand_addr_words = set(w for w in cand_row.get('address_words', cand_addr.split()) if len(w) > 1)

            # 1. Name word overlap & Jaccard
            common_words = s1_words.intersection(cand_words)
            union_words = s1_words.union(cand_words)
            common_words_count = len(common_words)
            word_jaccard = common_words_count / len(union_words) if union_words else 0.0

            # 2. String exact & substring matches
            is_exact_match = int(s1_clean == cand_clean and len(s1_clean) > 0)
            is_sub = int((s1_clean in cand_clean or cand_clean in s1_clean) and len(s1_clean) > 0 and len(cand_clean) > 0)

            # 3. State match & existence verification
            cand_state = cand_row.get('state')
            cand_has_state = cand_row.get('has_state', 1 if pd.notna(cand_state) else 0)
            both_have_state = int(s1_has_state == 1 and cand_has_state == 1)
            state_match = int(both_have_state == 1 and pd.notna(s1_state) and s1_state == cand_state)

            # 4. House number extraction & match
            raw_cand_addr = cand_row.get('business_address', cand_addr)
            cand_hn, cand_has_hn = extract_house_number(raw_cand_addr)
            both_have_hn = int(s1_has_hn == 1 and cand_has_hn == 1)
            house_num_match = int(both_have_hn == 1 and pd.notna(s1_hn) and s1_hn == cand_hn)

            # 5. Address common words & Jaccard
            addr_common = s1_addr_words.intersection(cand_addr_words)
            addr_union = s1_addr_words.union(cand_addr_words)
            addr_common_count = len(addr_common)
            addr_jaccard = addr_common_count / len(addr_union) if addr_union else 0.0

            row_dict = {
                's1_id': s1_id,
                'candidate_id': cand_id,
                'source': cand_src,
                'blocking_score': cand_row.get('matching_score', 0),
                'common_words_count': common_words_count,
                'word_jaccard': round(word_jaccard, 3),
                'is_exact_core_match': is_exact_match,
                'is_substring_match': is_sub,
                'has_state_s1': s1_has_state,
                'has_state_cand': cand_has_state,
                'both_have_state': both_have_state,
                'state_match': state_match,
                'has_house_num_s1': s1_has_hn,
                'has_house_num_cand': cand_has_hn,
                'both_have_house_num': both_have_hn,
                'house_number_match': house_num_match,
                'address_common_words_count': addr_common_count,
                'address_word_jaccard': round(addr_jaccard, 3)
            }
            if gt_df is not None:
                row_dict['label'] = int(cand_id in true_targets)

            feature_rows.append(row_dict)

    return pd.DataFrame(feature_rows)

pair_features_df = compute_pair_features(s1_top10, blocked_cands_gpu, s2, s3, gt_df=gt)
print(f"✓ Generated preview feature matrix for {len(pair_features_df):,} candidate pairs!")
if 'label' in pair_features_df.columns:
    n_pos = int((pair_features_df['label'] == 1).sum())
    n_neg = int((pair_features_df['label'] == 0).sum())
    ratio_str = f"1:{n_neg/n_pos:.1f}" if n_pos > 0 else "N/A"
    print(f"Class Balance: {n_pos:,} Positive Matches (label=1) | {n_neg:,} Negative Pairs (label=0) | Ratio: {ratio_str}")

first_s1_id = s1_top10.iloc[0]['entity_id']
display(pair_features_df[pair_features_df['s1_id'] == first_s1_id].head(8))


### 9. Scaled Training Feature Generation (10,000 S1 Entities → ~150,000 Candidate Pairs)
Scales candidate blocking and pairwise feature extraction across **10,000 $S_1$ reference entities** against the full candidate pools ($S_2 + S_3$):
- Bounded Top-15 selection: $10,000 \times 15 = 150,000$ candidate pairs.
- Extracts all comparison features and attaches binary ground truth labels (`label = 1 / 0`).
- Caches the resulting training matrix to `data/processed/train_features_10k_v2.parquet` via PyArrow for instant reloads during GBDT model tuning.


In [ ]:
N_TRAIN_S1 = 10000
TOP_K = 15
S1_BATCH_SIZE = 100
CAND_CHUNK_SIZE = 100000

print("--- Preparing Scaled Training Feature Dataset (10,000 S1 Entities) ---")
print(f"Target Reference Entities : {N_TRAIN_S1:,} S1 records")
print(f"Candidates per Entity     : Top {TOP_K} (Total ~{N_TRAIN_S1 * TOP_K:,} Candidate Pairs)")
print(f"Candidate Pool Sources    : S2 ({len(s2):,} records) + S3 ({len(s3):,} records)")

# 1. Select first 10,000 S1 reference records
s1_train = s1.head(N_TRAIN_S1).copy().reset_index(drop=True)

# Ensure house numbers extracted for S1 training set
if 'house_number' not in s1_train.columns:
    print("Extracting house numbers for S1 reference records...")
    hn_tuples = [extract_house_number(addr) for addr in s1_train['business_address']]
    s1_train['house_number'] = [t[0] for t in hn_tuples]
    s1_train['has_house_number'] = [t[1] for t in hn_tuples]

# 2. Run GPU-Accelerated Candidate Blocking
t0_block = time.time()
blocked_cands_train, summary_df_train = block_candidates_gpu(
    s1_train, 
    s2, 
    s3, 
    s1_batch_size=S1_BATCH_SIZE, 
    cand_chunk_size=CAND_CHUNK_SIZE, 
    top_k=TOP_K
)
t_block_elapsed = time.time() - t0_block
print(f"\n✓ GPU Candidate Blocking completed in {t_block_elapsed:.1f}s ({t_block_elapsed/60:.2f} mins)!")

# 3. Extract Pairwise Features & Attach Ground Truth Labels
t0_feat = time.time()
print(f"\nExtracting pairwise features for ~{N_TRAIN_S1 * TOP_K:,} candidate pairs...")
train_features_df = compute_pair_features(
    s1_train, 
    blocked_cands_train, 
    s2, 
    s3, 
    gt_df=gt
)
t_feat_elapsed = time.time() - t0_feat
print(f"✓ Feature extraction completed in {t_feat_elapsed:.2f}s!")

# 4. Display Dataset Dimensions & Class Distribution
print("\n" + "=" * 70)
print("TRAINING FEATURE DATASET READY FOR MODEL TRAINING")
print("=" * 70)
print(f"Total Candidate Pairs Generated : {len(train_features_df):,}")
print(f"Total Features per Pair        : {train_features_df.shape[1]} (comparison features + binary 'label')")

if 'label' in train_features_df.columns:
    n_pos = int((train_features_df['label'] == 1).sum())
    n_neg = int((train_features_df['label'] == 0).sum())
    imb_ratio = n_neg / n_pos if n_pos > 0 else 0
    print(f"Positive Matches (label = 1)   : {n_pos:,} ({n_pos / len(train_features_df) * 100:.2f}%)")
    print(f"Negative Pairs   (label = 0)   : {n_neg:,} ({n_neg / len(train_features_df) * 100:.2f}%)")
    print(f"Class Imbalance Ratio          : 1 : {imb_ratio:.1f} (use scale_pos_weight={imb_ratio:.1f} in LightGBM)")
print("=" * 70)

# 5. Save to disk as Parquet for fast reuse
os.makedirs("data/processed", exist_ok=True)
train_features_path = "data/processed/train_features_10k_v2.parquet"
try:
    train_features_df.to_parquet(train_features_path, index=False)
    print(f"✓ Successfully cached feature dataset to: {train_features_path}")
except Exception as e:
    csv_path = "data/processed/train_features_10k_v2.csv"
    train_features_df.to_csv(csv_path, index=False)
    print(f"✓ Cached feature dataset to CSV: {csv_path}")

display(train_features_df.head(10))


### 10. Supervised GBDT Training with Class Imbalance Compensation & Grouped Cross-Validation
Trains a high-performance **LightGBM Classifier** on the 150,000 candidate pairs generated in Step 9:
1. **Class Imbalance Compensation**: Configures `scale_pos_weight = 3.9` to balance positive vs negative gradients.
2. **5-Fold Grouped Cross-Validation (`GroupKFold` on `s1_id`)**: Strictly isolates all 15 candidate pairs of an $S_1$ entity to either the training or validation fold, completely preventing data leakage across folds.
3. **Engineered Interaction Features**: Enriches the feature space with interaction features (`name_x_addr_jaccard`, `both_exact_and_state`, `score_per_word`, `source_is_s3`).
4. **Optimal Decision Threshold Search**: Scans thresholds across out-of-fold predictions to find the decision cutoff that **maximizes F1 Score**.
5. **Out-of-Fold Metrics & Feature Importances**: Generates PR-AUC, ROC-AUC, Precision, Recall, F1 score, and feature importance rankings.


In [ ]:
# ==============================================================================
# 10. Supervised GBDT Training with Class Imbalance & Grouped Cross-Validation
# ==============================================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    f1_score, precision_score, recall_score, classification_report
)
from IPython.display import display

warnings.filterwarnings('ignore')

print("=" * 75)
print("TRAINING SUPERVISED GBDT MODEL WITH IMBALANCE COMPENSATION")
print("=" * 75)

# 1. Load or Verify Feature Dataset
if 'train_features_df' in globals() and isinstance(train_features_df, pd.DataFrame) and len(train_features_df) > 0:
    df_feat = train_features_df.copy()
    print(f"✓ Using in-memory feature dataset: {len(df_feat):,} candidate pairs")
else:
    parquet_path = "data/processed/train_features_10k_v2.parquet"
    csv_path = "data/processed/train_features_10k_v2.csv"
    if os.path.exists(parquet_path):
        df_feat = pd.read_parquet(parquet_path)
        print(f"✓ Loaded feature dataset from: {parquet_path} ({len(df_feat):,} pairs)")
    elif os.path.exists(csv_path):
        df_feat = pd.read_csv(csv_path)
        print(f"✓ Loaded feature dataset from: {csv_path} ({len(df_feat):,} pairs)")
    else:
        raise FileNotFoundError("Feature dataset not found! Please run Cell 20 first.")

# 2. Imbalance Inspection
n_pos = int((df_feat["label"] == 1).sum())
n_neg = int((df_feat["label"] == 0).sum())
scale_pos_weight = n_neg / max(1, n_pos)

print(f"\nClass Distribution:")
print(f"  • Positive Matches (label=1) : {n_pos:,} ({n_pos / len(df_feat) * 100:.2f}%)")
print(f"  • Negative Pairs   (label=0) : {n_neg:,} ({n_neg / len(df_feat) * 100:.2f}%)")
print(f"  • Imbalance Ratio            : 1 : {scale_pos_weight:.2f}")
print(f"  • Configured scale_pos_weight: {scale_pos_weight:.2f}")

# 3. Feature Engineering & Preparation
df_feat["source_is_s3"] = (df_feat["source"] == "S3").astype(int)
df_feat["name_x_addr_jaccard"] = df_feat["word_jaccard"] * df_feat["address_word_jaccard"]
df_feat["both_exact_and_state"] = df_feat["is_exact_core_match"] * df_feat["state_match"]
df_feat["score_per_word"] = df_feat["blocking_score"] / df_feat["common_words_count"].clip(lower=1)

FEATURE_COLS = [
    "blocking_score",
    "common_words_count",
    "word_jaccard",
    "is_exact_core_match",
    "is_substring_match",
    "has_state_s1",
    "has_state_cand",
    "both_have_state",
    "state_match",
    "has_house_num_s1",
    "has_house_num_cand",
    "both_have_house_num",
    "house_number_match",
    "address_common_words_count",
    "address_word_jaccard",
    "source_is_s3",
    "name_x_addr_jaccard",
    "both_exact_and_state",
    "score_per_word"
]

print(f"\nModel Training Features ({len(FEATURE_COLS)} features):")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:>2}. {f}")

# 4. 5-Fold GroupKFold Cross-Validation (Grouped strictly by s1_id to prevent data leakage)
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)
oof_preds = np.zeros(len(df_feat), dtype=np.float32)
models = []
feature_importances = np.zeros(len(FEATURE_COLS), dtype=np.float64)

X = df_feat[FEATURE_COLS]
y = df_feat["label"].values
groups = df_feat["s1_id"].values

print("\n" + "=" * 75)
print("STARTING 5-FOLD GROUPED CROSS-VALIDATION (Grouped by s1_id)")
print("=" * 75)

t0_train = time.time()
for fold, (trn_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    X_train, y_train = X.iloc[trn_idx], y[trn_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    fold_scale = (len(y_train) - y_train.sum()) / max(1, y_train.sum())
    
    clf = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=6,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=fold_scale,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    val_preds = clf.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    models.append(clf)
    feature_importances += clf.feature_importances_ / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_prauc = average_precision_score(y_val, val_preds)
    print(f"Fold {fold}/{N_SPLITS} | Best Iter: {clf.best_iteration_:>4d} | Val ROC-AUC: {fold_auc:.4f} | Val PR-AUC: {fold_prauc:.4f}")

t_train_elapsed = time.time() - t0_train
print(f"\n✓ 5-Fold Training completed in {t_train_elapsed:.2f}s ({t_train_elapsed/60:.2f} mins)!")

# 5. Out-of-Fold Evaluation & Threshold Tuning
overall_auc = roc_auc_score(y, oof_preds)
overall_prauc = average_precision_score(y, oof_preds)

print("\n" + "=" * 75)
print("OUT-OF-FOLD PERFORMANCE & THRESHOLD OPTIMIZATION (Maximizing F1)")
print("=" * 75)
print(f"Overall Out-of-Fold ROC-AUC : {overall_auc:.4f}")
print(f"Overall Out-of-Fold PR-AUC  : {overall_prauc:.4f}")

# Scan decision thresholds from 0.10 to 0.90
thresholds = np.linspace(0.10, 0.90, 81)
best_f1 = 0.0
best_thresh = 0.50
thresh_results = []

for th in thresholds:
    pred_binary = (oof_preds >= th).astype(int)
    p = precision_score(y, pred_binary, zero_division=0)
    r = recall_score(y, pred_binary, zero_division=0)
    f = f1_score(y, pred_binary, zero_division=0)
    thresh_results.append((th, p, r, f))
    if f > best_f1:
        best_f1 = f
        best_thresh = th

# Default threshold (0.50) metrics
def_p = precision_score(y, (oof_preds >= 0.50).astype(int), zero_division=0)
def_r = recall_score(y, (oof_preds >= 0.50).astype(int), zero_division=0)
def_f = f1_score(y, (oof_preds >= 0.50).astype(int), zero_division=0)

# Optimal threshold metrics
opt_p = precision_score(y, (oof_preds >= best_thresh).astype(int), zero_division=0)
opt_r = recall_score(y, (oof_preds >= best_thresh).astype(int), zero_division=0)
opt_f = best_f1

print("\nThreshold Comparison:")
print(f"  • Default Threshold (0.50)      : Precision={def_p*100:.2f}%, Recall={def_r*100:.2f}%, F1 Score={def_f*100:.2f}%")
print(f"  • Optimal Threshold ({best_thresh:.2f})      : Precision={opt_p*100:.2f}%, Recall={opt_r*100:.2f}%, F1 Score={opt_f*100:.2f}%")
print(f"  ↳ F1 Improvement: +{(opt_f - def_f)*100:.2f}%")

# 6. Detailed Classification Report at Optimal Threshold
print("\n" + "=" * 75)
print(f"DETAILED CLASSIFICATION REPORT (At Optimal Threshold = {best_thresh:.2f})")
print("=" * 75)
print(classification_report(y, (oof_preds >= best_thresh).astype(int), target_names=["Negative Pair (0)", "True Match (1)"], digits=4))

# 7. Feature Importance Ranking
feat_imp_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance": feature_importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("=" * 75)
print("TOP FEATURE IMPORTANCES (Averaged across 5 Folds)")
print("=" * 75)
display(feat_imp_df)

# 8. Save Trained Model Booster & Out-of-Fold Predictions
os.makedirs("models", exist_ok=True)
model_save_path = "models/lgbm_entity_resolver_v2.txt"
models[0].booster_.save_model(model_save_path)
print(f"\n✓ Saved trained model booster to: {model_save_path}")

df_feat["model_match_prob"] = oof_preds
df_feat["predicted_match"] = (oof_preds >= best_thresh).astype(int)
oof_save_path = "data/processed/oof_predictions_10k_v2.parquet"
try:
    df_feat[["s1_id", "candidate_id", "source", "label", "model_match_prob", "predicted_match"]].to_parquet(oof_save_path, index=False)
    print(f"✓ Cached out-of-fold predictions to: {oof_save_path}")
except Exception:
    csv_fallback = "data/processed/oof_predictions_10k_v2.csv"
    df_feat[["s1_id", "candidate_id", "source", "label", "model_match_prob", "predicted_match"]].to_csv(csv_fallback, index=False)
    print(f"✓ Cached out-of-fold predictions to CSV: {csv_fallback}")
